### 05 - Preparation des donnees de modelisation
#### HumanForYou - Attrition ML

Objectif: preparer train/test pour la prediction d'attrition **a partir de la sortie du notebook 04**.
Cette etape ne refait pas les traitements deja realises en 02/03/04 (imputation, feature engineering badgeuse, clustering).

- **Entree**: `data/processed/kmeans_clusters.csv`
- **Sorties**: `data/processed/attrition_train_prepared.csv`, `data/processed/attrition_test_prepared.csv`

#### 1. Imports

In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

#### 2. Chargement de la sortie 04

In [2]:
DATA_PATH = os.path.join('..', 'data', 'processed', 'kmeans_clusters.csv')
assert os.path.exists(DATA_PATH), f'Fichier introuvable: {DATA_PATH}'

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'NaN total: {df.isna().sum().sum()}')
df.head()

Shape: (4410, 28)
NaN total: 0


,Age,Attrition,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EmployeeID,Gender,JobLevel,...,YearsAtCompany,YearsSinceLastPromotion,YearsWithCurrManager,EnvironmentSatisfaction,JobSatisfaction,WorkLifeBalance,JobInvolvement,PerformanceRating,avg_work_hours,cluster
0,51,0,Travel_Rarely,Sales,6,2,Life Sciences,1,Female,1,...,1,0,0,3.0,4.0,2.0,3,3,7.373651,2
1,31,1,Travel_Frequently,Research & Development,10,1,Life Sciences,2,Female,1,...,5,1,4,3.0,2.0,4.0,2,4,7.718969,0
2,32,0,Travel_Frequently,Research & Development,17,4,Other,3,Male,4,...,5,0,3,2.0,2.0,1.0,3,3,7.013240,2
3,38,0,Non-Travel,Research & Development,2,5,Life Sciences,4,Male,3,...,8,7,5,4.0,4.0,3.0,2,3,7.193678,1
4,32,0,Travel_Rarely,Research & Development,10,1,Medical,5,Male,1,...,6,0,4,4.0,1.0,3.0,3,3,8.006175,2


#### 3. Verifications de continuite

In [3]:
required_cols = ['Attrition', 'avg_work_hours', 'cluster']
missing = [c for c in required_cols if c not in df.columns]
assert not missing, f'Colonnes attendues manquantes: {missing}'
assert df.isna().sum().sum() == 0, 'Des NaN subsistent alors que 02/03/04 ont deja nettoye les donnees.'
assert set(df['Attrition'].unique()).issubset({0, 1}), 'Attrition doit etre binaire 0/1 (sortie du 02).'
print('Validation des pre-traitements precedents: OK')

Validation des pre-traitements precedents: OK


#### 4. Split + encodage minimal (sans retraitement)

In [4]:
X = df.drop(columns=['Attrition']).copy()
y = df['Attrition'].astype(int).copy()

# EmployeeID est un identifiant technique, non utilise pour la prediction.
if 'EmployeeID' in X.columns:
    X = X.drop(columns=['EmployeeID'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Encodage categoriel necessaire pour les modeles supervises.
X_train_enc = pd.get_dummies(X_train, drop_first=False)
X_test_enc = pd.get_dummies(X_test, drop_first=False)
X_train_enc, X_test_enc = X_train_enc.align(X_test_enc, join='outer', axis=1, fill_value=0)

train_prepared_df = X_train_enc.copy()
train_prepared_df['Attrition'] = y_train.to_numpy()
test_prepared_df = X_test_enc.copy()
test_prepared_df['Attrition'] = y_test.to_numpy()

print(f'train_prepared_df: {train_prepared_df.shape}')
print(f'test_prepared_df : {test_prepared_df.shape}')

train_prepared_df: (3528, 47)
test_prepared_df : (882, 47)


#### 5. Export

In [5]:
PROCESSED_DIR = os.path.join('..', 'data', 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

train_output_path = os.path.join(PROCESSED_DIR, 'attrition_train_prepared.csv')
test_output_path = os.path.join(PROCESSED_DIR, 'attrition_test_prepared.csv')

train_prepared_df.to_csv(train_output_path, index=False)
test_prepared_df.to_csv(test_output_path, index=False)

print(f'Saved: {train_output_path}')
print(f'Saved: {test_output_path}')

Saved: ..\data\processed\attrition_train_prepared.csv
Saved: ..\data\processed\attrition_test_prepared.csv


In [6]:
assert train_prepared_df.isna().sum().sum() == 0, 'NaN dans train_prepared_df'
assert test_prepared_df.isna().sum().sum() == 0, 'NaN dans test_prepared_df'
assert 'Attrition' in train_prepared_df.columns and 'Attrition' in test_prepared_df.columns
assert 'cluster' in train_prepared_df.columns, 'La feature cluster issue du 04 doit etre presente.'
print('Validation OK pour 05_Regression_Preparation.ipynb')

Validation OK pour 05_Regression_Preparation.ipynb
